## Aplicando etapa de pré-processamento

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Adiciona a raiz do projeto ao PATH, caso necessário

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.preprocessing.preprocessing import (
    separar_variaveis,
    identificar_tipos_colunas,
    dividir_dados,
    preprocessar_dados
)

In [2]:
# Caminho da base final produzida na etapa de EDA
CAMINHO_BASE = ROOT / "data" / "alunos_com_meta_V2.parquet"

# Nome da variável target
TARGET = "alfabetizado_alunos"

# Variáveis utilizadas como granularidade para imputação
ID_MUNICIPIO = "id_municipio_alunos"
ID_UF = "id_uf"
REDE = "rede_alunos"

# Parâmetros da seleção de features
THRESHOLD_PEARSON = 0.10
THRESHOLD_SPEARMAN = 0.20
ALPHA_CHI2 = 0.05

print(f"Base: {CAMINHO_BASE}")

Base: c:\Users\monef\OneDrive\Documentos\GitHub\tech-challenge-fase3\data\alunos_com_meta_V2.parquet


In [3]:
df = pd.read_parquet(CAMINHO_BASE)

print(f"Dimensões da base: {df.shape}")

display(df.head())

Dimensões da base: (1791053, 14)


,id_municipio_alunos,rede_alunos,alfabetizado_alunos,nome_rede_alunos,taxa_alfabetizacao_2023,capital_uf,id_uf,nome_regiao,meta_alfabetizacao_2024,percentual_participacao_2024,indice_analfabetismo_2022,rendimento_domiciliar_2022,tipo_localizacao,inse_2023
0,2101301,3,0,Municipal,67.67,0.0,21,Nordeste,69.66,91.38,19.07,404.00,MESCLA,4.0709
1,2704302,3,0,Municipal,39.23,1.0,27,Nordeste,45.58,84.26,8.42,837.33,MESCLA,4.5752
2,4322509,3,0,Municipal,65.96,0.0,43,Sul,68.24,82.01,2.99,1237.33,MESCLA,5.2615
3,3502002,3,0,Municipal,44.62,0.0,35,Sudeste,50.32,88.33,4.45,1210.00,MESCLA,5.5267
4,4106902,3,0,Municipal,70.41,1.0,41,Sul,71.93,79.42,1.53,1850.00,MESCLA,5.5055


In [4]:
# Criando a variável de Atingiu Meta: 1 se atingiu ou superou a meta de 2024, 0 caso contrário
df['meta_atingida'] = (
    df['taxa_alfabetizacao_2023'] >= df['meta_alfabetizacao_2024']
).astype(int)

In [5]:
print(df['meta_atingida'].value_counts(normalize=True) * 100)

meta_atingida
0    91.994542
1     8.005458
Name: proportion, dtype: float64


In [6]:
print("Tipos das variáveis:")
display(df.dtypes.to_frame("dtype"))

print("\nQuantidade de nulos por variável:")
display(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .to_frame("nulos")
)

print("\nDistribuição da target:")
display(
    df[TARGET]
      .value_counts(dropna=False)
      .to_frame("quantidade")
)

print("\nPercentual da target:")
display(
    (df[TARGET]
       .value_counts(normalize=True, dropna=False)
       .mul(100)
       .round(2)
       .to_frame("percentual"))
)

Tipos das variáveis:


,dtype
id_municipio_alunos,int64
rede_alunos,int64
alfabetizado_alunos,int64
nome_rede_alunos,object
taxa_alfabetizacao_2023,float64
capital_uf,float64
id_uf,int64
nome_regiao,object
meta_alfabetizacao_2024,float64
percentual_participacao_2024,float64



Quantidade de nulos por variável:


,nulos
meta_alfabetizacao_2024,14617
percentual_participacao_2024,14617
inse_2023,1563
tipo_localizacao,1563
id_municipio_alunos,0
rede_alunos,0
alfabetizado_alunos,0
id_uf,0
capital_uf,0
taxa_alfabetizacao_2023,0



Distribuição da target:


,quantidade
alfabetizado_alunos,
1,941691
0,849362



Percentual da target:


,percentual
alfabetizado_alunos,
1,52.58
0,47.42


In [7]:
df = df.dropna()

print("\nQuantidade de nulos por variável:")
display(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .to_frame("nulos")
)


Quantidade de nulos por variável:


,nulos
id_municipio_alunos,0
rede_alunos,0
alfabetizado_alunos,0
nome_rede_alunos,0
taxa_alfabetizacao_2023,0
capital_uf,0
id_uf,0
nome_regiao,0
meta_alfabetizacao_2024,0
percentual_participacao_2024,0


In [8]:
len(df)

1774886

In [9]:
# ============================================================
# ANÁLISE DE OUTLIERS - TABELA COMPLETA
# ============================================================

colunas_numericas = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

colunas_numericas = [
    coluna for coluna in colunas_numericas
    if coluna not in ["id_municipio_alunos", "id_uf", "rede_alunos", "alfabetizado_alunos", "tipo_localizacao", "capital_uf", "meta_atingida"]
]

print(colunas_numericas)

resultados_outliers = []

for coluna in colunas_numericas:

    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = df[
        (df[coluna] < limite_inferior) |
        (df[coluna] > limite_superior)
    ]

    qtd_outliers = len(outliers)
    pct_outliers = (qtd_outliers / len(df)) * 100

    resultados_outliers.append({
        "coluna": coluna,
        "qtd_outliers": qtd_outliers,
        "pct_outliers": round(pct_outliers, 2),
        "limite_inferior": round(limite_inferior, 2),
        "limite_superior": round(limite_superior, 2),
        "min": round(df[coluna].min(), 2),
        "max": round(df[coluna].max(), 2)
    })

resumo_outliers = pd.DataFrame(resultados_outliers).sort_values(
    "pct_outliers",
    ascending=False
)

resumo_outliers

['taxa_alfabetizacao_2023', 'meta_alfabetizacao_2024', 'percentual_participacao_2024', 'indice_analfabetismo_2022', 'rendimento_domiciliar_2022', 'inse_2023']


,coluna,qtd_outliers,pct_outliers,limite_inferior,limite_superior,min,max
3,indice_analfabetismo_2022,129451,7.29,-7.70,20.78,0.90,34.68
2,percentual_participacao_2024,29706,1.67,71.45,103.05,70.00,100.00
0,taxa_alfabetizacao_2023,24969,1.41,14.92,96.92,4.35,100.00
1,meta_alfabetizacao_2024,6423,0.36,25.44,94.21,7.94,80.00
4,rendimento_domiciliar_2022,1460,0.08,-137.24,2062.07,228.57,2500.00
5,inse_2023,20,0.00,3.60,6.25,3.62,6.28


Para nosso pré processamento iremos deixar a coluna indice_analfabetismo_2022 com o método robust, por possuir uma % consideravel de outliers, o restante entra com o método minmaxscaler por possuir uma % pequena de outliers

In [10]:
# removendo a variavel "tipo_localizacao", pois possui apenas um valor dentro dela, tornando inutil 

df = df.drop(columns="tipo_localizacao")

In [11]:
X, y = separar_variaveis(
    df=df,
    target=TARGET
)

print(f"X: {X.shape}")
print(f"y: {y.shape}")

X: (1774886, 13)
y: (1774886,)


In [12]:
(
    X_train,
    X_test,
    y_train,
    y_test
) = dividir_dados(
    X=X,
    y=y,
    test_size=0.20,
    random_state=42
)

print("Dimensões:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

print(f"\ny_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

Dimensões:
X_train: (1419908, 13)
X_test:  (354978, 13)

y_train: (1419908,)
y_test:  (354978,)


In [13]:
distribuicao_conjuntos = pd.DataFrame({
    "Train": y_train.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True)
}).T.mul(100).round(2)

display(distribuicao_conjuntos)

alfabetizado_alunos,1,0
Train,52.72,47.28
Test,52.72,47.28


In [14]:
colunas_numericas, colunas_categoricas = identificar_tipos_colunas(X_train)

In [15]:
print(colunas_numericas)

['id_municipio_alunos', 'rede_alunos', 'taxa_alfabetizacao_2023', 'capital_uf', 'id_uf', 'meta_alfabetizacao_2024', 'percentual_participacao_2024', 'indice_analfabetismo_2022', 'rendimento_domiciliar_2022', 'inse_2023', 'meta_atingida']


In [16]:
print(colunas_categoricas)

['nome_rede_alunos', 'nome_regiao']


In [17]:
(
    X_train_processado,
    X_test_processado,
    y_train,
    y_test,
    preprocessor,
    features_selecionadas,
    correlacoes_pearson,
    resultados_spearman,
    resultados_chi2,
    distribuicao_target
) = preprocessar_dados(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    id_municipio=ID_MUNICIPIO,
    id_uf=ID_UF,
    rede=REDE,
    colunas_robust = ["indice_analfabetismo_2022"],
    colunas_minmax = [x for x in colunas_numericas if x not in ['id_municipio_alunos', 'rede_alunos', 'capital_uf', 'meta_atingida', 'id_uf', 'indice_analfabetismo_2022']],
    threshold_pearson=THRESHOLD_PEARSON,
    threshold_spearman=THRESHOLD_SPEARMAN,
    alpha_chi2=ALPHA_CHI2
)


Nulos após imputação:
Train: 0
Test: 0

PEARSON
taxa_alfabetizacao_2023         0.244812
meta_alfabetizacao_2024         0.236338
percentual_participacao_2024    0.185456
inse_2023                       0.034846
indice_analfabetismo_2022       0.034235
rendimento_domiciliar_2022      0.025138
dtype: float64

SPEARMAN
                   feature  correlacao  p_valor
0  taxa_alfabetizacao_2023    0.235276      0.0
1  meta_alfabetizacao_2024    0.234240      0.0

QUI-QUADRADO
            feature        chi2       p_valor
1       nome_regiao  187.125921  1.348047e-42
0  nome_rede_alunos         NaN           NaN

CHECK-UP DO DATASET FINAL

Dimensões:
Train: (1419908, 14)
Test:  (354978, 14)

Valores nulos:
Train: 0
Test:  0

Tipos das variáveis finais:
float64    14
Name: count, dtype: int64

DISTRIBUIÇÃO DA TARGET

Distribuição no TRAIN:
                     quantidade  percentual
alfabetizado_alunos                        
0                        671320   47.279119
1                    

In [18]:
print("Features selecionadas:")

print("\nPearson:")
print(features_selecionadas["pearson"])

print("\nSpearman:")
print(features_selecionadas["spearman"])

print("\nQui-quadrado:")
print(features_selecionadas["chi2"])

Features selecionadas:

Pearson:
['taxa_alfabetizacao_2023', 'meta_alfabetizacao_2024', 'percentual_participacao_2024']

Spearman:
['taxa_alfabetizacao_2023', 'meta_alfabetizacao_2024']

Qui-quadrado:
['nome_regiao']


In [19]:
print("=" * 60)
print("CHECK-UP FINAL")
print("=" * 60)

print("\nDimensões:")
print(f"Train: {X_train_processado.shape}")
print(f"Test:  {X_test_processado.shape}")

print("\nNulos:")
print(f"Train: {X_train_processado.isnull().sum().sum()}")
print(f"Test:  {X_test_processado.isnull().sum().sum()}")

print("\nTipos:")
display(X_train_processado.dtypes.value_counts())

print("\nPrimeiras linhas:")
display(X_train_processado.head())

CHECK-UP FINAL

Dimensões:
Train: (1419908, 14)
Test:  (354978, 14)

Nulos:
Train: 0
Test:  0

Tipos:


float64    14
Name: count, dtype: int64


Primeiras linhas:


,numericas_minmax__taxa_alfabetizacao_2023,numericas_minmax__meta_alfabetizacao_2024,numericas_minmax__percentual_participacao_2024,numericas_minmax__rendimento_domiciliar_2022,numericas_minmax__inse_2023,numericas_robust__indice_analfabetismo_2022,categoricas__nome_rede_alunos_Municipal,categoricas__nome_regiao_Centro-Oeste,categoricas__nome_regiao_Nordeste,categoricas__nome_regiao_Norte,categoricas__nome_regiao_Sudeste,categoricas__nome_regiao_Sul,passthrough__capital_uf,passthrough__meta_atingida
135001,0.529430,0.710519,0.462000,0.298239,0.557046,0.005626,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1063012,0.378881,0.539134,0.311333,0.102913,0.206501,1.578059,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
216055,0.265029,0.400638,0.447667,0.059623,0.039759,0.174402,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
353765,0.793623,1.000000,0.917667,0.076793,0.105975,3.651195,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1685207,0.702771,0.901471,0.650667,0.178196,0.389402,1.587904,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [20]:
# Resumo dos objetos disponíveis para o próximo notebook

objetos_modelagem = {
    "X_train": X_train_processado,
    "X_test": X_test_processado,
    "y_train": y_train,
    "y_test": y_test,
    "preprocessor": preprocessor,
    "features_selecionadas": features_selecionadas
}

print("Objetos preparados para a etapa de modelagem:")
print(list(objetos_modelagem.keys()))

Objetos preparados para a etapa de modelagem:
['X_train', 'X_test', 'y_train', 'y_test', 'preprocessor', 'features_selecionadas']


In [21]:
import joblib

joblib.dump(
    objetos_modelagem,
    "../data/objetos_modelagem.pkl"
)

print("Objetos salvos com sucesso!")

Objetos salvos com sucesso!
